# Rooftop Segmentation + Area Calculation  (v2 — Smart Filters)
## Zero-Shot Pipeline using SAM · Vegetation / Water / Texture filtering

**Problem fixed in v2:**
- SAM at 500 px → 25 detections (trees, grass, cars, pool included ❌)
- SAM at 5000 px → 1 detection (misses smaller roofs ❌)

**v2 solution — 5 additional filters on top of size:**

| Filter | What it removes |
|---|---|
| **Vegetation** (green dominance) | Lawns, trees — green channel >> red channel |
| **Water / pool** (blue dominance) | Swimming pools, ponds |
| **Color uniformity** (std dev) | Textured surfaces: tree canopies, grass |
| **Solidity** (convex hull ratio) | Spiky tree crowns, irregular blobs |
| **Rectangularity** (bbox fill) | Roads, diagonal/curved shapes |

**Sweet spot MIN_AREA_PX = 2000** (was 500 → noise; 5000 → misses roofs).

Area formula: `area_m² = pixel_count × GSD²`


## Step 0: Install Dependencies

In [ ]:
!pip install segment-anything -q
!pip install opencv-python-headless shapely -q

import os
SAM_CKPT = '/content/sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O /content/sam_vit_b_01ec64.pth
    print('SAM checkpoint downloaded.')
else:
    print('SAM checkpoint already present.')

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Configuration

In [ ]:
import os, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# ─────────────────────────────────────────────────────────────
#  CONFIGURE BEFORE RUNNING
# ─────────────────────────────────────────────────────────────
CFG = {
    # Paths
    'IMAGE_DIR'    : '/content/drive/MyDrive/project/images',
    'METADATA_CSV' : '/content/drive/MyDrive/project/metadata.csv',
    'OUTPUT_DIR'   : '/content/drive/MyDrive/project/rooftop_results',

    # SAM model
    'SAM_CKPT'     : '/content/sam_vit_b_01ec64.pth',
    'SAM_TYPE'     : 'vit_b',   # 'vit_b' fast | 'vit_l'/'vit_h' accurate (needs A100)

    # ── GSD (Ground Sample Distance) ──────────────────────────
    # GSD = metres/pixel.  area_m² = pixels × GSD²
    #   AIRS / Google Maps z=21 → 0.075 m/px
    #   Google Maps z=20       →  0.15 m/px
    #   Drone at 50 m          → ~0.015 m/px
    #   Set None → areas in pixels² only
    'GSD'          : 0.075,   # ← CHANGE THIS

    # ── SIZE FILTERS ──────────────────────────────────────────
    # Experiment showed: 500px = 25 detections (noise), 5000px = 1 detection (too strict)
    # 2000px is the sweet spot for AIRS-resolution imagery.
    'MIN_AREA_PX'  : 2000,    # min pixels  ← tweak if needed (see tuning guide)
    'MAX_AREA_FRAC': 0.35,    # max fraction of image area (sky / ground)

    # ── SHAPE FILTERS ─────────────────────────────────────────
    'MIN_STABILITY': 0.85,    # SAM confidence [0-1]; leaves/textures score lower
    'MIN_RECT'     : 0.45,    # mask_area / bbox_area; roads & L-blobs score low
    'MIN_SOLIDITY' : 0.70,    # mask_area / convex_hull_area; tree crowns ~ 0.4-0.6

    # ── COLOR FILTERS (new in v2) ─────────────────────────────
    # Vegetation (lawns, tree canopies): green channel >> red channel
    'MAX_GREEN_DOMINANCE': 15,  # mean(G) - mean(R) > this → skip
    # Pool / water: blue channel >> red and green
    'MAX_BLUE_DOMINANCE' : 20,  # mean(B) - mean(R) > this AND mean(B)-mean(G) > this → skip
    # Rooftops are uniform in colour; tree canopies are not
    'MAX_COLOR_STD'      : 45,  # std-dev of pixel values inside mask > this → skip

    # ── SAM GENERATION ────────────────────────────────────────
    'POINTS_PER_SIDE' : 32,   # increase to 64 for very dense urban images
    'PRED_IOU_THRESH' : 0.86,
    'BOX_NMS_THRESH'  : 0.50,

    # Processing
    'RESIZE'      : 1024,     # long-edge resize for SAM input
    'BATCH_LIMIT' : None,     # e.g. 50 to test on first 50 images
}

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(os.path.join(CFG['OUTPUT_DIR'], 'masks'),    exist_ok=True)
os.makedirs(os.path.join(CFG['OUTPUT_DIR'], 'overlays'), exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'GSD    : {CFG["GSD"]} m/px' if CFG['GSD'] else 'GSD not set — areas in pixels²')
print(f'MIN_AREA_PX : {CFG["MIN_AREA_PX"]}')

## Step 3: Load SAM Model

In [ ]:
sam = sam_model_registry[CFG['SAM_TYPE']](checkpoint=CFG['SAM_CKPT'])
sam.to(DEVICE)

mask_generator = SamAutomaticMaskGenerator(
    model           = sam,
    points_per_side = CFG['POINTS_PER_SIDE'],
    pred_iou_thresh = CFG['PRED_IOU_THRESH'],
    box_nms_thresh  = CFG['BOX_NMS_THRESH'],
    output_mode     = 'binary_mask',
)
print('SAM model loaded and ready.')

## Step 4: Smart Rooftop Filters + Area Functions (v2)

### Why each filter is needed

| Filter | Example of what it removes |
|---|---|
| `MIN_AREA_PX = 2000` | Parked cars (~800 px), pavement cracks |
| `MIN_STABILITY = 0.85` | Low-confidence leaf/texture masks |
| `MIN_RECT = 0.45` | Diagonal road sections, L-shaped blobs |
| `MIN_SOLIDITY = 0.70` | Tree crowns (spiky outline → low hull fill) |
| `MAX_GREEN_DOMINANCE = 15` | Lawns (G ≈ 140, R ≈ 100 → diff = 40 > 15) |
| `MAX_BLUE_DOMINANCE = 20` | Swimming pools (B ≈ 160, R ≈ 80 → diff = 80 > 20) |
| `MAX_COLOR_STD = 45` | Grass patches (high texture variance ~55-70) |

Rooftops have: low color variance, uniform texture, solid outline, and are roughly rectangular.


In [ ]:
import cv2, numpy as np

def resize_for_sam(image_np, max_size=1024):
    h, w = image_np.shape[:2]
    scale = max_size / max(h, w)
    if scale < 1.0:
        image_np = cv2.resize(image_np, (int(w*scale), int(h*scale)),
                              interpolation=cv2.INTER_LINEAR)
    return image_np, (h, w)


def rectangularity(mask_bool):
    """mask_area / bounding_box_area.  Perfect rectangle = 1.0."""
    rows = np.any(mask_bool, axis=1)
    cols = np.any(mask_bool, axis=0)
    if not rows.any():
        return 0.0
    rmin, rmax = np.where(rows)[0][[0,-1]]
    cmin, cmax = np.where(cols)[0][[0,-1]]
    bbox_area = max((rmax-rmin+1) * (cmax-cmin+1), 1)
    return float(mask_bool.sum()) / bbox_area


def solidity(mask_bool):
    """
    mask_area / convex_hull_area.
    Tree crowns: spiky outline → 0.4–0.6.
    Rooftops: solid shape    → 0.75–0.95.
    """
    msk = mask_bool.astype(np.uint8) * 255
    contours, _ = cv2.findContours(msk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0
    largest   = max(contours, key=cv2.contourArea)
    hull      = cv2.convexHull(largest)
    hull_area = cv2.contourArea(hull)
    if hull_area == 0:
        return 0.0
    return cv2.contourArea(largest) / hull_area


def color_stats(img_rgb, mask_bool):
    """Return (mean_R, mean_G, mean_B, overall_std) inside the mask."""
    pixels = img_rgb[mask_bool]   # (N, 3)
    if pixels.shape[0] == 0:
        return 0, 0, 0, 999
    return (float(pixels[:,0].mean()),
            float(pixels[:,1].mean()),
            float(pixels[:,2].mean()),
            float(pixels.std()))


def filter_rooftop_masks(sam_masks, image_shape, image_rgb=None):
    """
    Apply layered rooftop filters to SAM output.

    Parameters
    ----------
    sam_masks   : list of SAM mask dicts
    image_shape : (H, W) of the image SAM processed
    image_rgb   : numpy (H,W,3) RGB image — needed for color filters

    Returns
    -------
    (filtered_masks, reject_log)
    reject_log  : dict showing how many masks each filter removed
    """
    H, W      = image_shape[:2]
    img_area  = H * W
    results   = []
    log = {'size':0,'stability':0,'rect':0,'solidity':0,
           'green':0,'blue':0,'std':0}

    for m in sam_masks:
        area = int(m['area'])
        stab = float(m['stability_score'])
        mask = m['segmentation']         # bool (H, W)

        # 1. Size
        if area < CFG['MIN_AREA_PX'] or area > CFG['MAX_AREA_FRAC'] * img_area:
            log['size'] += 1; continue

        # 2. Stability
        if stab < CFG['MIN_STABILITY']:
            log['stability'] += 1; continue

        # 3a. Rectangularity
        rect = rectangularity(mask)
        if rect < CFG['MIN_RECT']:
            log['rect'] += 1; continue

        # 3b. Solidity (convex hull fill)
        sol = solidity(mask)
        if sol < CFG['MIN_SOLIDITY']:
            log['solidity'] += 1; continue

        # 4 & 5. Color filters (optional — only if image supplied)
        if image_rgb is not None:
            mr, mg, mb, cstd = color_stats(image_rgb, mask)

            # 4a. Vegetation: green >> red
            if mg - mr > CFG['MAX_GREEN_DOMINANCE']:
                log['green'] += 1; continue

            # 4b. Water / pool: blue >> red AND blue >> green
            if (mb - mr > CFG['MAX_BLUE_DOMINANCE'] and
                mb - mg > CFG['MAX_BLUE_DOMINANCE']):
                log['blue'] += 1; continue

            # 5. Colour uniformity: rooftops are homogeneous
            if cstd > CFG['MAX_COLOR_STD']:
                log['std'] += 1; continue

        out          = dict(m)
        out['rectangularity'] = rect
        out['solidity']       = sol
        results.append(out)

    return results, log


def compute_area(pixel_count, gsd=None, scale_factor=1.0):
    """Convert pixel count to real-world area."""
    px = pixel_count * scale_factor
    if gsd is None:
        return {'pixels': px}
    m2 = px * gsd**2
    return {'pixels': px, 'area_m2': m2, 'area_ft2': m2 * 10.7639}


print('✅ v2 filter functions defined.')

## Step 5: Visualisation Helper

In [ ]:
COLORS = [
    [255, 80, 80],[80,200,80],[80,130,255],
    [255,200,50],[200,80,200],[50,220,220],
    [255,140,0],[180,255,80],[255,80,180],
]

def visualise_rooftops(image_np, rooftop_masks, gsd=None, title='', save_path=None):
    overlay       = image_np.copy().astype(np.float32)
    combined_mask = np.zeros(image_np.shape[:2], dtype=np.uint8)
    legend_patches = []

    for i, m in enumerate(rooftop_masks):
        color = COLORS[i % len(COLORS)]
        mask  = m['segmentation']
        area  = compute_area(int(m['area']), gsd)

        for ci, cv in enumerate(color):
            overlay[..., ci] = np.where(
                mask,
                overlay[..., ci] * 0.45 + cv * 0.55,
                overlay[..., ci])

        label = (f'Roof {i+1}: {area["area_m2"]:.1f} m²' if gsd
                 else f'Roof {i+1}: {int(area["pixels"])} px²')
        combined_mask[mask] = 255
        legend_patches.append(
            mpatches.Patch(color=[v/255 for v in color], label=label))

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(image_np);  axes[0].set_title('Original Image'); axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title(
        f'Detected Rooftops ({len(rooftop_masks)} found)\nGSD = {gsd} m/px' if gsd
        else f'Detected Rooftops ({len(rooftop_masks)} found)')
    axes[1].axis('off')
    if legend_patches:
        axes[1].legend(handles=legend_patches, loc='lower left',
                       fontsize=7, ncol=max(1, len(legend_patches)//8))
    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    return combined_mask

print('Visualisation helper defined.')

## Step 6: Single Image Test (run this first!)

Run on ONE image, check the `reject_log` printout, tune thresholds in Step 2, repeat.


In [ ]:
# ── Pick a test image ────────────────────────────────────────────
# Option A: first image in CSV
df_meta = pd.read_csv(CFG['METADATA_CSV'])
test_name = df_meta.iloc[0]['image_name']
test_path = os.path.join(CFG['IMAGE_DIR'], test_name)

# Option B: set manually
# test_path = '/content/drive/MyDrive/project/images/BWZEK3E9OJAOY.png'
# test_name = os.path.basename(test_path)

# ── Load ─────────────────────────────────────────────────────────
bgr = cv2.imread(test_path)
assert bgr is not None, f'Cannot read: {test_path}'
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

orig_hw = rgb.shape[:2]
small, _ = resize_for_sam(rgb, max_size=CFG['RESIZE'])
scale_factor = (orig_hw[0] * orig_hw[1]) / (small.shape[0] * small.shape[1])

print(f'Image        : {test_path}')
print(f'Original     : {orig_hw[1]}×{orig_hw[0]}')
print(f'SAM input    : {small.shape[1]}×{small.shape[0]}')

# ── Run SAM ──────────────────────────────────────────────────────
print('\nRunning SAM...')
all_masks = mask_generator.generate(small)
print(f'SAM generated {len(all_masks)} raw masks.')

# ── Filter with v2 smart filters (pass image for color filters) ──
rooftop_masks, reject_log = filter_rooftop_masks(all_masks, small.shape, small)
print(f'\nAfter v2 filters: {len(rooftop_masks)} rooftop candidates.')
print()
print('── Reject log (how many masks each filter removed) ──')
for k, v in reject_log.items():
    reasons = {
        'size'     : 'too small (<2000px) or too large (>35% image)',
        'stability': 'SAM low confidence (<0.85)',
        'rect'     : 'not rectangular enough (<0.45)',
        'solidity' : 'spiky/irregular outline (<0.70) — likely tree crown',
        'green'    : 'green-dominant pixels — vegetation (lawn/trees)',
        'blue'     : 'blue-dominant pixels — water/pool',
        'std'      : 'high colour variance — textured surface (not uniform roof)',
    }
    print(f'  {k:12s}: {v:3d}  ({reasons[k]})')

# ── Visualise ─────────────────────────────────────────────────────
save_path = os.path.join(CFG['OUTPUT_DIR'], 'overlays',
                         os.path.splitext(test_name)[0] + '_rooftops_v2.png')
visualise_rooftops(small, rooftop_masks, gsd=CFG['GSD'],
                   title=test_name, save_path=save_path)

# ── Area table ────────────────────────────────────────────────────
print(f'\n{"─"*55}')
print(f'{"Roof #":<8} {"Pixels":>10} {"Area (m²)":>12} {"Area (ft²)":>12}  {"Rect":>6}  {"Solid":>6}')
print(f'{"─"*55}')
total_m2 = 0.0
for i, m in enumerate(rooftop_masks):
    a   = compute_area(int(m['area']), CFG['GSD'], scale_factor)
    m2  = a.get('area_m2',  float('nan'))
    ft2 = a.get('area_ft2', float('nan'))
    total_m2 += m2 if not np.isnan(m2) else 0
    print(f'{i+1:<8} {int(a["pixels"]):>10,} {m2:>12.1f} {ft2:>12.1f}  '
          f'{m["rectangularity"]:>6.3f}  {m["solidity"]:>6.3f}')
print(f'{"─"*55}')
print(f'{"TOTAL":>19} {total_m2:>12.1f} m²  ({total_m2 * 10.7639:.1f} ft²)')

## 🔧 Tuning Guide

Use the `reject_log` printout from Step 6 to guide adjustments.

| `reject_log` key too high? | Problem | Fix |
|---|---|---|
| `size` | Missing small roofs | Lower `MIN_AREA_PX` → 1500 or 1000 |
| `size` | Ground still detected | Lower `MAX_AREA_FRAC` → 0.20 |
| `stability` | Valid roofs rejected | Lower `MIN_STABILITY` → 0.80 |
| `rect` | Shadow-split roofs rejected | Lower `MIN_RECT` → 0.38 |
| `solidity` | L-shaped roofs rejected | Lower `MIN_SOLIDITY` → 0.60 |
| `green` | Dark-green roofs rejected | Raise `MAX_GREEN_DOMINANCE` → 20–25 |
| `blue` | Blue roofs rejected | Raise `MAX_BLUE_DOMINANCE` → 30 |
| `std` | Tiled/patterned roofs rejected | Raise `MAX_COLOR_STD` → 55–65 |
| None removed | Trees/grass still in | Lower `MAX_GREEN_DOMINANCE` → 10, 8 |
| None removed | Pool still in | Lower `MAX_BLUE_DOMINANCE` → 12 |

> 💡 **Expected for BWZEK3E9OJAOY.png:** With v2 defaults you should see the 2–3 main building
> rooftops (red/terracotta) detected. Green garden areas, the blue pool, and parked cars should
> all be excluded now.

Workflow:
1. Run Step 6 → read reject_log
2. Adjust the biggest number's threshold in Step 2
3. Re-run Step 6 only (don't need to reload model)
4. When happy → run Step 7 (batch)


## Step 7: Batch Processing — All Images

In [ ]:
from tqdm.notebook import tqdm

df_meta = pd.read_csv(CFG['METADATA_CSV'])
if CFG['BATCH_LIMIT']:
    df_meta = df_meta.head(CFG['BATCH_LIMIT'])

results = []

for _, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc='Processing images'):
    img_name = row['image_name']
    img_path = os.path.join(CFG['IMAGE_DIR'], img_name)

    bgr = cv2.imread(img_path)
    if bgr is None:
        print(f'[SKIP] {img_name}'); continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    orig_hw = rgb.shape[:2]
    small, _ = resize_for_sam(rgb, max_size=CFG['RESIZE'])
    sf = (orig_hw[0]*orig_hw[1]) / (small.shape[0]*small.shape[1])

    try:
        all_masks, _log = mask_generator.generate(small), {}
        rooftop_masks, _log = filter_rooftop_masks(all_masks, small.shape, small)
    except Exception as e:
        print(f'[ERROR] {img_name}: {e}'); continue

    overlay_path = os.path.join(CFG['OUTPUT_DIR'], 'overlays',
                                os.path.splitext(img_name)[0] + '_rooftops.png')
    combined_mask = visualise_rooftops(small, rooftop_masks, gsd=CFG['GSD'],
                                       title=img_name, save_path=overlay_path)

    mask_path = os.path.join(CFG['OUTPUT_DIR'], 'masks',
                             os.path.splitext(img_name)[0] + '_mask.png')
    Image.fromarray(combined_mask).save(mask_path)

    image_total_m2 = 0.0
    for i, m in enumerate(rooftop_masks):
        a   = compute_area(int(m['area']), CFG['GSD'], sf)
        m2  = a.get('area_m2',  None)
        ft2 = a.get('area_ft2', None)
        image_total_m2 += (m2 or 0)
        results.append({
            'image_name'         : img_name,
            'roof_index'         : i + 1,
            'area_pixels'        : int(a['pixels']),
            'area_m2'            : round(m2,  2) if m2  else None,
            'area_ft2'           : round(ft2, 2) if ft2 else None,
            'stability'          : round(float(m['stability_score']), 4),
            'rectangularity'     : round(float(m['rectangularity']),  4),
            'solidity'           : round(float(m['solidity']),         4),
            'n_roofs_in_image'   : len(rooftop_masks),
            'total_roof_area_m2' : round(image_total_m2, 2) if CFG['GSD'] else None,
        })

results_df = pd.DataFrame(results)
csv_path   = os.path.join(CFG['OUTPUT_DIR'], 'rooftop_areas.csv')
results_df.to_csv(csv_path, index=False)

print(f'\n✅ Done. Processed {df_meta["image_name"].nunique()} images.')
print(f'   Total rooftops detected : {len(results_df)}')
print(f'   Results CSV             : {csv_path}')

## Step 8: Summary Statistics

In [ ]:
csv_path   = os.path.join(CFG['OUTPUT_DIR'], 'rooftop_areas.csv')
results_df = pd.read_csv(csv_path)

print('=== Per-Roof Area Distribution ===')
print(results_df['area_m2'].describe().round(2))

print('\n=== Per-Image Summary ===')
per_image = results_df.groupby('image_name').agg(
    n_roofs       = ('roof_index', 'count'),
    total_area_m2 = ('area_m2',    'sum'),
    avg_roof_m2   = ('area_m2',    'mean'),
).round(2).reset_index()
print(per_image.head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(results_df['area_m2'].dropna(), bins=40,
             color='steelblue', edgecolor='white')
axes[0].set_xlabel('Roof Area (m²)');  axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Individual Roof Areas')

axes[1].hist(per_image['n_roofs'],
             bins=range(1, int(per_image['n_roofs'].max())+2),
             color='coral', edgecolor='white', align='left')
axes[1].set_xlabel('Rooftops per Image');  axes[1].set_ylabel('Count')
axes[1].set_title('Rooftops per Image')

plt.tight_layout()
plt.savefig(os.path.join(CFG['OUTPUT_DIR'], 'area_distribution.png'), dpi=150)
plt.show()
print(f'\nTotal rooftop area : {results_df["area_m2"].sum():.1f} m²')
print(f'Average roof size  : {results_df["area_m2"].mean():.1f} m²')